# Train the gas-cylinder detector on Google Colab

This notebook fine-tunes YOLOv8n on the CylinDeRS dataset using a free Colab GPU,
which is much faster than training on a laptop CPU.

Before you start:
1. Upload `cylinders_dataset.zip` to the root of your Google Drive (**My Drive**).
2. Check that **Runtime → Change runtime type** is set to **T4 GPU**.

Then run every cell from top to bottom with **Runtime → Run all** and keep this tab open.

## 1. Check the GPU

`nvidia-smi` should show a **Tesla T4**. If it says `command not found`, this runtime has
no GPU: choose **Runtime → Change runtime type → T4 GPU** and run the notebook again.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

The same version as `requirements.txt` in the repository.

In [ ]:
!pip install -q "ultralytics~=8.4.171"

## 3. Connect Google Drive

Colab asks for permission to access your Drive. The dataset is read from Drive, and the
training results are written back to Drive, so they are kept even if Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Unzip the dataset

The zip is copied to the Colab disk first, because training straight from Drive is very slow.
At the end you should see `data.yaml`, `test`, `train` and `valid`.

In [ ]:
# Change this path if you put the zip in another Drive folder.
DATASET_ZIP = "/content/drive/MyDrive/cylinders_dataset.zip"

!cp "{DATASET_ZIP}" /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/data
!ls /content/data

## 5. Train

Same pre-trained model and image size as `src/model.py`, but with the full dataset on the GPU.

- `time=0.75` stops training after 45 minutes. Ultralytics measures the first epoch and then
  trains as many epochs as fit. Increase it if you have more time and want a better model.
- `patience=10` stops earlier when the validation score has not improved for 10 epochs.
- After every epoch, the weights are saved to `My Drive/gas_cylinders_runs/colab/weights/`.

In [ ]:
from ultralytics import YOLO

# Load the pre-trained nano model, the same one src/model.py starts from.
model = YOLO("yolov8n.pt")
model.train(
    data="/content/data/data.yaml",
    project="/content/drive/MyDrive/gas_cylinders_runs",  # Save to Drive, not to the Colab disk.
    name="colab",
    time=0.75,  # Train for at most 45 minutes; this overrides epochs.
    patience=10,  # Stop after 10 epochs without validation improvement.
    imgsz=480,  # Same image size as src/model.py.
    batch=32,  # The GPU has enough memory for bigger batches than a laptop.
    cache=True,  # Keep the images in RAM, because Colab has few CPUs to read the JPGs.
)
print(f"Training results: {model.trainer.save_dir}")
print(f"Best weights: {model.trainer.best}")

## 6. Look at the results

`results.png` shows the losses and the validation scores after every epoch.
The losses should go down, while precision, recall and mAP should go up.

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{model.trainer.save_dir}/results.png"))

## 7. Use the model on your laptop

1. In Google Drive, open `gas_cylinders_runs/colab/weights/` and download `best.pt`.
   If you trained more than once, the folder is `colab2`, `colab3`, ... (cell 5 prints the exact path).
2. Put it in the repository as `runs/colab/weights/best.pt`. The `runs/` folder is ignored by Git.
3. From the repository root, evaluate it on the test split and track the cylinders in the video:

```bash
python src/evaluate.py --weights runs/colab/weights/best.pt
python src/track.py --weights runs/colab/weights/best.pt
```

**If Colab disconnects during training**, run cells 1-4 again, then continue from the last epoch:

```python
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/gas_cylinders_runs/colab/weights/last.pt")
model.train(resume=True)
```